# 56 - GOI's own ranking vs. ours, evaluated head-to-head against the same ground truth

Notebook 55 showed GOI's ranking and ours *differ* (overlap, coverage gap, rank correlation). This notebook asks the sharper question: which one is actually *better* at finding truly relevant companies, measured against ground truth neither system controls.

**Avoiding circularity**: scoring our deployment classifier (trained on all gold labels) against those same gold labels would be circular, it already knows some of the answers. Instead this uses leave-one-query-out cross-validation, same principle as notebook 38, just extended from 5 to all 101 queries: for every query, train on every *other* query's gold labels, then rank that query's full candidate pool with a model that never saw its labels. Only 15 seconds for all 101 folds, so no cost concern.

**k-sweep, not a single cutoff**: Recall@k/Precision@k/NDCG@k computed at k = 10, 50, 100, 300, 500, 1000 (same k values as `27_baseline_colbert.ipynb`), for both GOI's own ranking (`production_results.xlsx`) and ours, against the same gold-verified relevant set (`gold_label == 2`).

**A caveat worth stating upfront**: most of the 101 queries still only have ~10-14 gold labels (Section on the notebook-40 expansion), so their per-query Recall is a coarse, noisy measurement, one company found or missed swings it by a large margin. The 19 deeply-labelled queries (5 original pilot + 14 headline queries, ~70-90 gold labels each) give a much more statistically stable read, so both an all-101-query table and a deep-queries-only table are reported.

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import LeaveOneGroupOut

OUTPUT_DIR = Path("result/56_goi_vs_ours_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_THRESHOLD = 2
K_VALUES = [10, 50, 100, 300, 500, 1000]
DEEP_QUERY_IDS = [1, 2, 3, 4, 5, 11, 12, 15, 34, 14, 27, 66, 72, 99, 101, 56, 82, 91, 92]  # pilot + headline

feature_cols = ["score_minilm", "score_linq", "score_gte", "score_bm25",
                "invrank_minilm", "invrank_linq", "invrank_gte", "invrank_bm25",
                "reranker_score", "n_channels"]

base_gold = pd.read_json("result/40_active_learning_labeling_queue/expanded_gold_labels.json")
round_gold = pd.read_json("result/42_headline_query_deepening/round_gold_labels.json")
features = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")
production = pd.read_excel("dataset/production_results.xlsx")[["query_id", "domain", "rank"]].rename(columns={"rank": "production_rank"})

gold = pd.concat([
    base_gold[["query_id", "domain", "gold_label"]],
    round_gold[["query_id", "domain", "gold_label"]],
], ignore_index=True).drop_duplicates(subset=["query_id", "domain"])

labeled = gold.merge(features, on=["query_id", "domain"], how="inner")
labeled["relevant"] = (labeled["gold_label"] >= RELEVANT_THRESHOLD).astype(int)
print(f"Gold labels for cross-validation: {len(labeled)} across {labeled['query_id'].nunique()} queries")

relevant_sets = {
    qid: set(g.loc[g["gold_label"] >= RELEVANT_THRESHOLD, "domain"])
    for qid, g in labeled.groupby("query_id")
}
print(f"Queries with at least one gold-verified relevant company: {sum(1 for v in relevant_sets.values() if v)}/101")

In [ ]:
X = labeled[feature_cols].values
y = labeled["relevant"].values
groups = labeled["query_id"].values
logo = LeaveOneGroupOut()

t0 = time.time()
our_rankings = {}  # query_id -> list of domains, ranked by CV score, top-1000

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
    held_out_query = groups[test_idx][0]
    clf = HistGradientBoostingClassifier(max_iter=150, max_depth=4, class_weight="balanced", random_state=0)
    clf.fit(X[train_idx], y[train_idx])

    query_candidates = features[features["query_id"] == held_out_query]
    scores = clf.predict_proba(query_candidates[feature_cols].values)[:, 1]
    ranked = query_candidates.assign(cv_score=scores).sort_values("cv_score", ascending=False)
    our_rankings[held_out_query] = ranked["domain"].head(1000).tolist()

    if (fold + 1) % 25 == 0 or (fold + 1) == 101:
        print(f"  {fold+1}/101 queries cross-validated")

print(f"Done in {time.time()-t0:.1f}s -- every query ranked by a model that never saw its own gold labels")

In [ ]:
goi_rankings = {
    qid: g.sort_values("production_rank")["domain"].tolist()
    for qid, g in production.groupby("query_id")
}


def precision_at_k(retrieved, relevant, k):
    if k == 0 or not relevant:
        return 0.0
    return len(set(retrieved[:k]) & relevant) / k


def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return None  # can't measure recall with zero known-relevant companies for this query
    return len(set(retrieved[:k]) & relevant) / len(relevant)


def dcg_at_k(retrieved, relevant, k):
    return sum(1 / np.log2(i + 2) for i, d in enumerate(retrieved[:k]) if d in relevant)


def ndcg_at_k(retrieved, relevant, k):
    if not relevant:
        return None
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0.0


def evaluate(rankings, query_ids, label):
    rows = []
    for qid in query_ids:
        retrieved = rankings.get(qid, [])
        relevant = relevant_sets.get(qid, set())
        for k in K_VALUES:
            rows.append({
                "system": label, "query_id": qid, "k": k,
                "precision": precision_at_k(retrieved, relevant, k),
                "recall": recall_at_k(retrieved, relevant, k),
                "ndcg": ndcg_at_k(retrieved, relevant, k),
            })
    return pd.DataFrame(rows)


eval_goi = evaluate(goi_rankings, list(goi_rankings.keys()), "GOI (production)")
eval_ours = evaluate(our_rankings, list(our_rankings.keys()), "Ours (cross-validated)")
eval_all = pd.concat([eval_goi, eval_ours], ignore_index=True)
eval_all.to_csv(OUTPUT_DIR / "per_query_metrics.csv", index=False)
print(f"Evaluated {eval_all['query_id'].nunique()} queries for both systems -> {OUTPUT_DIR / 'per_query_metrics.csv'}")

In [ ]:
def summarize(df, title):
    print(f"=== {title} ===")
    summary = df.groupby(["system", "k"])[["precision", "recall", "ndcg"]].mean().round(3)
    print(summary)
    print()
    return summary


all_summary = summarize(eval_all, f"All 101 queries (n={eval_all['query_id'].nunique()})")

deep_mask = eval_all["query_id"].isin(DEEP_QUERY_IDS)
deep_summary = summarize(eval_all[deep_mask], f"Deep-labelled queries only (n={eval_all.loc[deep_mask, 'query_id'].nunique()}, ~70-90 gold labels each)")

all_summary.to_csv(OUTPUT_DIR / "summary_all_queries.csv")
deep_summary.to_csv(OUTPUT_DIR / "summary_deep_queries.csv")

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, (df, title) in zip(axes, [(eval_all, "All 101 queries"), (eval_all[deep_mask], "Deep-labelled queries only")]):
        recall_curve = df.groupby(["system", "k"])["recall"].mean().unstack("system")
        for system in recall_curve.columns:
            ax.plot(recall_curve.index, recall_curve[system], marker="o", label=system)
        ax.set_xlabel("k")
        ax.set_ylabel("Recall@k")
        ax.set_title(title)
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plot_path = OUTPUT_DIR / "recall_at_k_comparison.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Saved -> {plot_path}")
except ImportError:
    print("matplotlib not available -- skipping the plot, the CSV tables above still have the same numbers.")